In [33]:
import cobra
import escher

In [34]:
# Load the model
model = cobra.io.read_sbml_model("model.xml")

In [35]:
# Save the path to the Escher map (with glycolysis and TCA cycle)
amac_map_path = "escher/MIT1002_glycolysis_and_tca_escher-map.json"

In [36]:
# Set the medium
minimal_media = {
    "EX_cpd00027_e0": 10,  # Glucose_e0
    "EX_cpd00007_e0": 20,  # O2_e0
    "EX_cpd00058_e0": 1000,  # Cu2+_e0
    "EX_cpd00971_e0": 1000,  # Na+_e0
    "EX_cpd00063_e0": 1000,  # Ca2+_e0
    "EX_cpd00048_e0": 1000,  # Sulfate_e0
    "EX_cpd10516_e0": 1000,  # fe3_e0
    "EX_cpd00254_e0": 1000,  # Mg_e0
    "EX_cpd00009_e0": 1000,  # Phosphate_e0
    "EX_cpd00205_e0": 1000,  # K+_e0
    "EX_cpd00013_e0": 1000,  # NH3_e0
    "EX_cpd00099_e0": 1000,  # Cl-_e0
    "EX_cpd00030_e0": 1000,  # Mn2+_e0
    "EX_cpd00075_e0": 1000,  # Nitrite_e0
    "EX_cpd00001_e0": 1000,  # H2O_e0
    "EX_cpd00034_e0": 1000,  # Zn2+_e0
    "EX_cpd00149_e0": 1000,  # Co2+_e0
}
# Set the model medium
model.medium = minimal_media

Could not identify an external compartment by name and choosing one with the most boundary reactions. That might be complete nonsense or change suddenly. Consider renaming your compartments using `Model.compartments` to fix this.


In [37]:
# Run normal FBA
fba_solution =model.optimize()
fba_solution

,fluxes,reduced_costs
rxn02201_c0,0.004390,-1.387779e-17
rxn00351_c0,0.000000,-4.471342e-02
rxn07431_c0,0.000000,1.639310e-16
rxn00836_c0,0.000000,-2.235671e-02
rxn00423_c0,0.000000,-5.551115e-17
...,...,...
EX_cpd00039_e0,0.000000,-1.168646e-01
EX_cpd00054_e0,0.000000,-4.522153e-02
rxn15341_c0,0.000000,-4.922261e-17
rxn01519_c0,0.005749,2.775558e-17


In [38]:
# Run pFBA
pfba_solution = cobra.flux_analysis.pfba(model)
pfba_solution

,fluxes,reduced_costs
rxn02201_c0,0.004390,-2.000000
rxn00351_c0,0.000000,316.666667
rxn07431_c0,0.000000,-2.000000
rxn00836_c0,0.000000,163.333333
rxn00423_c0,0.000000,-2.000000
...,...,...
EX_cpd00039_e0,0.000000,843.333333
EX_cpd00054_e0,0.000000,311.333333
rxn15341_c0,0.000000,-2.000000
rxn01519_c0,0.005749,-2.000000


In [39]:
pfba_solution.fluxes["bio1_biomass"]  # Glucose exchange reaction flux

0.9145926901670548

In [40]:
# Make map with the pFBA solution
builder = escher.Builder(
    model=model, map_json=amac_map_path
)
builder.reaction_data = pfba_solution.fluxes

In [41]:
builder

Builder(reaction_data={'rxn02201_c0': 0.0043901564815146815, 'rxn00351_c0': 0.0, 'rxn07431_c0': 0.0, 'rxn00836…

In [42]:
# Look for specifc reactions
pfba_solution.fluxes["rxn00875_c0"]  # Butanoyl-CoA:Acetate CoA-transferase (in the acetate --> Acetyl-CoA loop)

-2.209299598841892e-14

In [43]:
pfba_solution.fluxes["rxn00097_c0"]  # Nucleotide balancing reaction that had infinite flux

1.42900813757753

In [44]:
pfba_solution.fluxes["rxn00260_c0"]  # L-aspartate reaction that made a loop in the TCA cycle

5.292642378936855

In [45]:
model.reactions.rxn00260_c0

Reaction identifier,rxn00260_c0
Name,L-Aspartate:2-oxoglutarate aminotransferase [c0]
Memory address,0x7f9c61b87bb0
Stoichiometry,cpd00024_c0 + cpd00041_c0 <=> cpd00023_c0 + cpd00032_c0 2-Oxoglutarate + L-Aspartate <=> L-Glutamate + Oxaloacetate
GPR,WP_049587271.1
Lower bound,-1000.0
Upper bound,1000.0


In [46]:
# Knock of the EMP-pathway specific reaction (phosphofructokinase)
model.reactions.rxn00545_c0.knock_out()
# Rerun pFBA
pfba_no_pfk = cobra.flux_analysis.pfba(model)
# Make map with the pFBA solution
builder = escher.Builder(
    model=model, map_json=amac_map_path
)
builder.reaction_data = pfba_no_pfk.fluxes
builder

Builder(reaction_data={'rxn02201_c0': 0.004390156481514548, 'rxn00351_c0': 0.0, 'rxn07431_c0': 0.0, 'rxn00836_…

In [47]:
model.metabolites.cpd00222_c0

Metabolite identifier,cpd00222_c0
Name,GLCN
Memory address,0x7f9c4a5fc850
Formula,C6H11O7
Compartment,c0
In 2 reaction(s),"rxn01121_c0, rxn01275_c0"


In [48]:
model.reactions.rxn01275_c0

Reaction identifier,rxn01275_c0
Name,ATP:D-Gluconate 6-phosphotransferase [c0]
Memory address,0x7f9c7201c8e0
Stoichiometry,cpd00002_c0 + cpd00222_c0 --> cpd00008_c0 + cpd00067_c0 + cpd00284_c0 ATP + GLCN --> ADP + H+ + 6-Phospho-D-gluconate
GPR,WP_039216382.1
Lower bound,0.0
Upper bound,1000.0


In [49]:
model.reactions.rxn01121_c0

Reaction identifier,rxn01121_c0
Name,D-gluconate hydro-lyase (2-dehydro-3-deoxy-D-gluconate-forming) [c0]
Memory address,0x7f9c4abf2d90
Stoichiometry,cpd00222_c0 --> cpd00001_c0 + cpd00176_c0 GLCN --> H2O + 2-keto-3-deoxygluconate
GPR,WP_014975676.1
Lower bound,0.0
Upper bound,1000.0


In [51]:
# Add a source reaction for gluconate
model.add_boundary(model.metabolites.cpd00222_c0, type="sink")
# Set the lower bound of the gluconate exchange reaction to allow uptake
model.reactions.SK_cpd00222_c0.lower_bound = -10
# Remove knock out of phosphofructokinase
model.reactions.rxn00545_c0.upper_bound = 1000
# Remove glucose from the medium
glc_free_medium = model.medium.copy()
glc_free_medium["EX_cpd00027_e0"] = 0
model.medium = glc_free_medium
# Rerun pFBA
pfba_gluconate = cobra.flux_analysis.pfba(model)
# Make map with the pFBA solution
builder = escher.Builder(
    model=model, map_json=amac_map_path
)
builder.reaction_data = pfba_gluconate.fluxes
builder

Could not identify an external compartment by name and choosing one with the most boundary reactions. That might be complete nonsense or change suddenly. Consider renaming your compartments using `Model.compartments` to fix this.
Could not identify an external compartment by name and choosing one with the most boundary reactions. That might be complete nonsense or change suddenly. Consider renaming your compartments using `Model.compartments` to fix this.


Builder(reaction_data={'rxn02201_c0': 0.003997819703820076, 'rxn00351_c0': 0.0, 'rxn07431_c0': 0.0, 'rxn00836_…

In [ ]:
pfba_gluconate.fluxes["SK_cpd00222_c0"]  # Gluconate sink reaction flux

-1000.0